# 高级检索与重排（Advanced Retrieval & Reranking）


## 粗召回与精重排——Cross-Encoder 重排机制
1. 核心学习目标
    1. 理解痛点：掌握为何单阶段向量检索（Bi-Encoder）无法满足高精准度需求。
    2. 底层原理：深挖 Bi-Encoder 与 Cross-Encoder 在 Transformer 注意力机制（Self-Attention）层面的根本差异。
    3. 架构设计：掌握生产级 RAG 的“粗召回（Recall） $\rightarrow$ 精重排（Rerank）”两阶段pipeline。
    4. 学会使用主流重排模型（如 `bge-reranker`）完成打分与重排序。

2. 为什么需要“粗召回 + 精重排”两阶段架构？
    * 单阶段向量检索的局限：向量模型（Bi-Encoder）将一段文本压缩为一个固定维度的向量（如 768 或 1536 维）。这种压缩不可避免会丢失局部细粒度语义（如否定词、数量词、专业术语间的微弱联系）。只用向量检索，经常召回“语义看起来很像，但关键事实不匹配”的文档。
    * 两阶段架构解法：
        * 第一阶段：粗召回（Recall）
            * 目标：从百万/千万级文档库中，快速找回可能相关的 Top 50 ~ 100 篇文档。
            * 手段：混合检索（BM25 关键词 + Bi-Encoder 向量检索）。
            * 特点：毫秒级响应，宁可多召回（高召回率），不漏掉关键信息。

        * 第二阶段：精重排（Rerank）
            * 目标：对召回的 50 ~ 100 篇文档逐一精细打分，剔除噪声，选出最准确的 Top 3 ~ 5 篇喂给 LLM。
            * 手段：Cross-Encoder（交叉编码器）。
            * 特点：深度语义交叉，高精准度，计算开销较大，但仅作用于少量候选集。

3. Bi-Encoder 与 Cross-Encoder 深度机制对比

    | 维度    | Bi-Encoder（双编码器 / Embedding 模型）                                        | Cross-Encoder（交叉编码器 / Reranker）                                                |
    |-------|------------------------------------------------------------------------|--------------------------------------------------------------------------------|
   | 典型代表  | `BGE-Large-ZH`, OpenAI `text-embedding-3`                              | `BGE-Reranker-Large`, `Cohere Rerank`                                          |
   | 计算架构  | Query 和 Doc 分别输入模型，提取各自独立向量，通过余弦相似度打分。                                 | Query 和 Doc 拼接在一起 `[CLS] Query [SEP] Doc [SEP]` 共同输入模型。                        |
   | 注意力交互 | Late Interaction（延迟/无交互）,Query 与 Doc 的 Token 在 Transformer 内部没有全量交叉关注。 | Full Self-Attention（全量交叉交互）,Query 的每个 Token 与 Doc 的每个 Token 互相计算 Attention 权重。 |
   | 计算效率  | 极高,  Doc 向量可提前离线计算并存入向量库，查询时仅做纯向量点积/余弦计算。                              | 较低, 无法预先计算 Doc 向量，每次查询都需进行深度 Transformer 前向传播。                                 |
   | 准确率   | 中等（易受向量压缩的信息损失影响）。                                                     | 极高（能精准捕捉否定、对比、细节修饰等逻辑关系）。                                                      |


基于 `bge-reranker` 实现精重排,在工程落地中，我们常用 Hugging Face 的 `sentence-transformers` 或 `FlagEmbedding` 来调用开源 Reranker 模型。

In [ ]:
from sentence_transformers import CrossEncoder

# 1. 加载开箱即用的重排模型（以智源开源的 bge-reranker-large 为例）
model_name = "BAAI/bge-reranker-large"
reranker = CrossEncoder(model_name, max_length=512)

# 2. 模拟第一阶段粗召回（Recall）得到的候选文档列表（Top 5）
query = "公司员工年假未休完，是否可以结转到下一年？"

documents = [
    "文档A：根据公司规章，员工年假应在当年休完。因工作原因未能休完的，经部门经理批准可延至次年3月前休完。",
    "文档B：员工婚假为3天，产假为128天，休假期间工资照发。",
    "文档C：员工因公出差可申请差旅补贴，标准按照《差旅管理制度》执行。",
    "文档D：年假标准为：工作满1年不满10年的，年休假5天；满10年不满20年的，年休假10天。",
    "文档E：未休年假的工资报酬，按员工日工资收入的300%支付，其中包含正常工作期间的工资。"
]

# 3. 构造 Cross-Encoder 的输入格式：[(Query, Doc1), (Query, Doc2), ...]
pair_inputs = [[query, doc] for doc in documents]

# 4. 计算重排得分（得分越高，相关性强；通常评分区间未标准化，可进行相对排序）
scores = reranker.predict(pair_inputs)

# 5. 将文档与得分组合并按分数降序排列
ranked_results = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)

print("=== 两阶段检索之精重排（Rerank）结果 ===")
for rank, (doc, score) in enumerate(ranked_results, start=1):
    print(f"Rank {rank} (Score: {score:.4f}): {doc}\n")


#### 生产级 RAG 中的 Reranker 工程优化技巧
1. Top-K 截断策略：
    * 粗召回阶段（BM25 + Dense Vector）：建议召回 Top 50 ~ Top 100。
    * 精重排阶段（Cross-Encoder）：重排后截取 Top 3 ~ Top 5 传给 LLM 上下文。

2. 性能与延迟平衡（Latency Budget）：
    * 如果 Reranker 导致 API 延迟过高（> 300ms），可选用较小参数量的模型（如 `bge-reranker-base`），或使用 ONNX Runtime / TensorRT 进行模型推理加速。

3. 阈值过滤（Score Thresholding）：
    * 设置硬性得分门槛（如得分低于某一标准时拒答或返回空），可以极大地降低 LLM 的“幻觉（Hallucination）”率。


1. 思考题：为什么我们不能直接用 Cross-Encoder 对向量数据库里的全量 100 万条文档进行直接检索，而必须先经过 Bi-Encoder 粗召回？
2. 动手练习：尝试使用week7-01中的混合检索系统，在最后接上 Cross-Encoder 代码，对比“混合检索单独输出”与“混合检索 + Rerank 重排”的答案相关度差异。

## 长上下文迷失与上下文压缩机制
两阶段检索（粗召回 + Cross-Encoder 重排）之后，我们检索到了相关度极高的 Top-K 文档片段。然而，直接将这些文档拼接并一股脑塞给 LLM，往往会导致新的工程痛点：Token 费用暴涨、推理延迟增加，甚至大模型出现“看了后面忘了中间”的信息遗忘现象。所以我们将聚焦于 RAG 检索管线（Pipeline）送入大模型之前的临门一脚——长上下文优化与上下文压缩（Context Compression & Lost in the Middle Solution）。

1. 理解痛点：深刻剖析大模型注意力机制带来的 “Lost in the Middle”（中间迷失） 现象及其危害。
2. 底层原理：掌握上下文重排序（Context Reordering）与语义压缩（Context Compression）的技术方案。
3. 架构设计：构建“召回 $\rightarrow$ 重排 $\rightarrow$ 过滤压缩/重排位置 $\rightarrow$ 喂给 LLM”的高效 RAG 上下文优化流。
4. 使用 Python 实现文档位置重组（Long Context Reorder）与基于 LLM/提炼模型的上下文压缩。


#### 什么是 “Lost in the Middle” 现象？
* 现象：当给 LLM 提供的 Prompt 上下文很长（包含多个检索到的文档 Chunk）时，LLM 对位于 Prompt 最开头（Beginning） 和 最结尾（End） 的信息提取准确率最高，而位于 中间（Middle） 的信息提取准确率显著下降（呈现 U 型曲线）。
* 成因：由于 Transformer 自注意力（Self-Attention）及 Decoder-Only 架构在生成新 Token 时，对序列两端的 Key/Value 注意力权重分布往往更高。
* RAG 中的隐患：如果 Reranker 输出的最佳文档（Top 1）被简单按顺序放在上百个 Chunk 的中间，LLM 极有可能直接忽略它，导致回答出现“幻觉”或拒答。


#### 上下文优化两大核心工程手段
为了解决“中间迷失”与“信息过载（Information Overload）”，生产环境通常配合采用以下两种策略：

1. 策略一：上下文重排（Long Context Reorder）
    * 策略机制：改变 Top-K 文档放置到 Prompt 中的物理顺序，把最相关的文档放在最前和最后，相关度稍低的放在中间。
    * 排序规则示例（假定得分高到低为 $D_1, D_2, D_3, D_4, D_5$）：
        * 原排序：$[D_1, D_2, D_3, D_4, D_5]$ （最关键的 $D_1, D_2$ 紧挨在一起或容易落在中前段）
        * 重排后：$[D_1, D_3, D_5, D_4, D_2]$ （最顶级的 $D_1$ 放在最前，$D_2$ 放在最后）

2. 策略二：上下文压缩（Context Compression）
    * 策略机制：检索到的文本块（如 512 Token）中往往只有 1-2 句话真正回答了 Query，其余是背景废话。通过轻量级模型或 Prompt 抽取，仅提取与 Query 直接强相关 的句子或 Token。
    * 代表方案：
        * 句/段落级抽取：使用小模型打分，裁剪无关段落。
        * Token 级压缩（如 LLMLingua / CompactPrompt）：利用小模型（如 GPT-2/Llama-small）计算 Token 的困惑度（Perplexity），删除信息量低、可预测的冗余词，压缩率可达 30%~50% 且几乎不损耗准确率。

**上下文重排与节点压缩，实战 1：实现 Long Context Reordering（防止中间迷失）**

In [ ]:
def long_context_reorder(documents: list) -> list:
    """
    将按相关度由高到低排列的文档列表，重新排列为“两端高相关、中间低相关”的序列。
    输入 documents: [Doc1(最高分), Doc2, Doc3, Doc4, Doc5(最低分)]
    输出 reordered: [Doc1, Doc3, Doc5, Doc4, Doc2]
    """
    reordered = []
    # 按照降序，偶数索引放左边，奇数索引倒序放右边
    for i, doc in enumerate(documents):
        if i % 2 == 0:
            reordered.insert(len(reordered) // 2, doc)
        else:
            reordered.append(doc)

    # 进一步保证得分最高的在最两侧
    # 简易两端分布算法：
    left = []
    right = []
    for i, doc in enumerate(documents):
        if i % 2 == 0:
            left.append(doc)
        else:
            right.insert(0, doc)
    return left + right

# 测试代码
docs = [
    "【相关度第1】公司员工年假未休完，经批准可结转至次年3月前休完。",
    "【相关度第2】年假结转需提前填写申请表，由部门经理及 HR 审批。",
    "【相关度第3】未在次年3月前休完的结转年假将自动清零，不予经济补偿。",
    "【相关度第4】公司员工每年享有 5-15 天的标准带薪年休假。",
    "【相关度第5】加班调休假与法定年假相互独立，优先消耗调休假。"
]

reordered_docs = long_context_reorder(docs)

print("=== 原始按相关度降序排列 ===")
for idx, d in enumerate(docs):
    print(f"位置 {idx}: {d}")

print("\n=== 重排后（优化 LLM 两端注意力） ===")
for idx, d in enumerate(reordered_docs):
    print(f"位置 {idx}: {d}")

**实战 2：语义上下文压缩（Context Extractor）**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

class SimpleContextCompressor:
    """基于相关性得分截取关键句子的上下文压缩器"""
    def __init__(self, threshold: float = 0.3):
        self.threshold = threshold

    def compress(self, query: str, document: str) -> str:
        # 将长文档按句号/换行切分为句子列表
        sentences = [s.strip() for s in document.split("。") if s.strip()]
        relevant_sentences = []

        for sentence in sentences:
            # 简化示例：按重叠词或轻量打分筛选（工程上可替换为 LLMLingua 或 MiniLM）
            overlap = set(query).intersection(set(sentence))
            score = len(overlap) / max(len(set(query)), 1)

            if score >= self.threshold:
                relevant_sentences.append(sentence)

        return "。".join(relevant_sentences) + "。" if relevant_sentences else document

# 使用示例
query = "年假未休完可以结转吗？"
long_doc = (
    "公司成立于2010年，位于科技园区。关于休假制度，公司员工年假未休完，经批准可结转至次年3月前休完。"
    "此外，公司提供免费午餐和健身房福利。员工应遵守考勤制度，不得无故迟到早退。"
)

compressor = SimpleContextCompressor(threshold=0.25)
compressed_doc = compressor.compress(query, long_doc)

print("原始文档长度:", len(long_doc))
print("压缩后文档内容:", compressed_doc)
print("压缩后文档长度:", len(compressed_doc))


#### 生产级 RAG 管线完整流向总结

1. 用户输入 $\rightarrow$ Query 改写/扩展/HyDE（解决表述不清）
2. 混合检索（BM25 关键词 + 向量检索） $\rightarrow$ 粗召回 Top 100
3. RRF 融合与重排（Cross-Encoder Reranker） $\rightarrow$ 精选 Top 10
4. 上下文压缩与重排（Long Context Reorder & Compression） $\rightarrow$ 剔除噪声、调整物理顺序为 Top 5
5. Prompt 拼接 $\rightarrow$ 输入 LLM 生成高质量回答


1. 思考题：在 RAG 中，如果上下文压缩过度（例如误删了带有否定词“不”、“非”的修饰成分），会导致什么后果？工程上应该如何设置安全边界？
    * 将直接导致语义反转与事实性错误
    * 工程上需通过硬约束保护、原文留底、压后校验及量化评估建立安全边界
2. 动手练习：尝试在你的 RAG 测试集中，对比“按 Reranker 分数直接拼接输入 LLM”与“经过 LongContextReorder 重排后拼接输入 LLM”，观察大模型在复杂推理问题上的回答准确度变化。